# 06 — Evaluar en holdout

No entrena. Mide el `.pt` que exportó el `05` contra el heurístico, mismo
validador, p=1 s=1, un contenedor.

- Holdout de producto (5 pedidos nunca vistos en train/val).
- Val de `data/scale` (los 20 que eligieron epoch; no es el examen final).

Métrica de **producto**: `volume_utilization`. La comparación vs PCT es el `09`.

Siguiente: `07_multipallet_firstfit.ipynb`.

In [1]:
from pathlib import Path
import json
import sys

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    ML_ROOT = HERE.parent
else:
    ML_ROOT = HERE if (HERE / "src_ml").is_dir() else Path("/home/edmundo/packing-services/online_policy_ml")
REPO_ROOT = ML_ROOT.parent
for p in (ML_ROOT / "src_ml", REPO_ROOT / "src"):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from bootstrap import setup
setup()

from evaluate import evaluate_orders, write_report
from methodology import assert_holdout_excluded, compare_paired
from paths import BED_JSON_NAME, DEMO_BED_BPP, ORDER_IDS_NAME, REPORTS_DIR, model_path_mlp_ppo_production, scale_split_dir
from splits import load_orders
from packing_services.online.learned.production import PRODUCT_HOLDOUT_ORDER_IDS

LOOKAHEAD_P, SELECT_S = 1, 1
ckpt = model_path_mlp_ppo_production(LOOKAHEAD_P, SELECT_S)
assert ckpt.is_file(), f"Falta el PPO del 05: {ckpt}"


def run_pair(name, orders, ids):
    rows_h = evaluate_orders(orders, ids, engine="heuristic", lookahead_p=LOOKAHEAD_P, select_s=SELECT_S)
    rows_rl = evaluate_orders(
        orders, ids, engine="learned", lookahead_p=LOOKAHEAD_P, select_s=SELECT_S, model_path=str(ckpt)
    )
    vs = compare_paired(rows_rl, rows_h, label_a="ppo", label_b="heuristic")
    print(name, "H", vs["mean_b"], "PPO", vs["mean_a"], "Δ", vs["mean_delta"], "IC", vs["ci_lo"], vs["ci_hi"])
    return {"name": name, "order_ids": ids, "heuristic": rows_h, "ppo": rows_rl, "vs_heuristic": vs}

holdout_orders = load_orders(DEMO_BED_BPP)
holdout_ids = sorted(PRODUCT_HOLDOUT_ORDER_IDS)
val_dir = scale_split_dir("val")
val_ids = json.loads((val_dir / ORDER_IDS_NAME).read_text(encoding="utf-8"))
assert_holdout_excluded(val_ids, role="evaluar_val")
payload = {
    "checkpoint": str(ckpt.relative_to(REPO_ROOT)),
    "holdout": run_pair("holdout", holdout_orders, holdout_ids),
    "scale_val": run_pair("scale_val", load_orders(val_dir / BED_JSON_NAME), val_ids),
}
write_report(REPORTS_DIR / "06_evaluar_holdout.json", payload)
print("Siguiente: 07_multipallet_firstfit.ipynb")

holdout H 0.6681564000000001 PPO 0.666782 Δ -0.0013743999999999978 IC -0.024781600000000004 0.019040400000000023
scale_val H 0.67442965 PPO 0.6874582 Δ 0.01302855000000001 IC -0.0002022587499999914 0.026794220000000007
Siguiente: 07_multipallet_firstfit.ipynb
